# SDA-DSC-212 — Time Series Analysis and Forecasting
## Lab 2 — The Stationarity Report

**Duration** 50 minutes · **Work in pairs** · `git checkout lab2-solution`

> The one rule for this whole course: every feature, every split and every scaler
> must be computable using only data that existed at the forecast origin.

### Tasks

| min | task |
|---|---|
| 8 | Run `stationarity_report` on the raw log-demand for 2023. Interpret ADF and KPSS **together**. Record the verdict and predict the differencing you will need. |
| 10 | `choose_differencing` with `m=24` — seasonal difference first, then regular. Print the trace and re-run the report to confirm stationarity was reached. |
| 10 | Plot before / after differencing. Check the lag-1 ACF for the over-differencing signature. Reduce `d` if you see it. |
| 12 | ACF / PACF with `lags >= 60`. Annotate the seasonal spike at lag 24. Read candidate `(p,q)` and seasonal `(P,Q)` off the fingerprint rules. |
| 10 | Write `STATIONARITY.md`: the verdict, the chosen `(d, D)`, and 2–3 candidate SARIMA orders with justification. Commit. |

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.figsize": (11, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25})

DATA = "../data"

Lab 1's output, reloaded — the labs are independent notebooks.

In [ ]:
from tayyar.data.load import load_demand, load_calendar, join_calendar

df = load_demand(f"{DATA}/ksa_grid_demand.csv")
df = join_calendar(df, load_calendar(f"{DATA}/ksa_calendar.csv"))

y = df["demand_mw"].astype(float)
ylog = np.log(y)          # variance first (Lab 1), mean second (this lab)
sub = ylog["2023"]        # one clean year — the tests stay fast and the plots stay readable

print(f"log-demand, 2023: {len(sub):,} hourly observations")
print(f"from {sub.index[0]} to {sub.index[-1]}")

### Task 1 — ADF and KPSS, read together

### The two tests have opposite nulls

This is the single most common error in the room, so read it slowly.

| test | H0 (the null) | small p (< 0.05) means | large p means |
|---|---|---|---|
| **ADF** | there **is** a unit root — *non-stationary* | reject H0 → **evidence of stationarity** | cannot reject a unit root |
| **KPSS** | the series **is** stationary | reject H0 → **evidence of non-stationarity** | cannot reject stationarity |

A small p-value is good news from ADF and bad news from KPSS. Reconciling them:

| | KPSS says stationary | KPSS says non-stationary |
|---|---|---|
| **ADF says stationary** | `stationary` — proceed | `difference-stationary` — difference once and re-test |
| **ADF says non-stationary** | `trend-stationary` — detrend, or add a trend regressor | `non-stationary` — difference and re-test |

The disagreement cells are not a failure of the method. They are the reason we run both:
they separate a series that needs differencing from one that needs detrending.

In [ ]:
from tayyar.analysis.stationarity import stationarity_report, format_report

r0 = stationarity_report(sub)
print(format_report(r0))
print()
print(f"ADF  p = {r0['adf_p']}  ->  reject 'unit root'      ->  evidence of STATIONARITY")
print(f"KPSS p = {r0['kpss_p']}  ->  reject 'is stationary'  ->  evidence of NON-STATIONARITY")

**Expected**

```
ADF  stat=  -3.103  p=0.0263  -> stationary
KPSS stat=   5.258  p=0.01    -> non-stationary
VERDICT: difference-stationary
ACTION : ADF stationary, KPSS not: a deterministic/level shift remains — difference once and re-test.
```

The tests disagree, and the disagreement is the finding. ADF can reject a unit root because
the daily cycle keeps pulling the series back towards its mean; KPSS still sees a level that
does not sit still across the year — summer runs far above winter. That combination is
`difference-stationary`: the remedy is a difference, not a detrend.

Note `kpss_p = 0.01`. KPSS p-values come from a lookup table and are clipped at the ends of
that table, so `0.01` means "at most 0.01" and `0.10` means "at least 0.10". statsmodels warns
about this in words; the warning is expected and is not an error.

> **Stop and predict.** You have a strongly seasonal hourly series with an annual swing.
> Write down the `(d, D)` you expect `choose_differencing` to return before you run the next
> cell. Most pairs write `d=1, D=1`. Keep your prediction — you will need it in a minute.

### Task 2 — the minimum differencing, seasonal first

In [ ]:
from tayyar.analysis.differencing import choose_differencing, difference

diffs = choose_differencing(sub, m=24)      # seasonal difference FIRST, then regular

cols = ["d", "D", "adf_stat", "adf_p", "kpss_stat", "kpss_p", "verdict"]
print(diffs["trace"][cols].to_string(index=False))
print()
print(f"chosen: d={diffs['d']}, D={diffs['D']}, m={diffs['m']}  ->  {diffs['final_verdict']}")
print()

ds = difference(sub, diffs["d"], diffs["D"], m=24)
print(format_report(stationarity_report(ds)))

**Expected**

```
 d  D  adf_stat  adf_p  kpss_stat  kpss_p               verdict
 0  0    -3.103 0.0263      5.258    0.01 difference-stationary
 0  1   -15.823 0.0000      0.092    0.10            stationary

chosen: d=0, D=1, m=24  ->  stationary
```

**`d = 0`, `D = 1`.** One seasonal difference was enough; no regular difference was needed.

This surprises almost everyone, including people who have fitted ARIMA models for years —
the reflex is `d=1, D=1`. Read the trace and see why the reflex is wrong: `y_t - y_{t-24}`
removes both the daily cycle *and* the slow level movement, because consecutive days sit at
almost the same level. Having removed both, there is nothing left for a regular difference
to remove — only structure the AR terms should be estimating.

The rule the routine encodes: **stop at the first stationary verdict.** Not "difference until
it looks nice", not "difference the standard amount".

### Task 3 — before / after, and the over-differencing check

In [ ]:
from statsmodels.tsa.stattools import acf

sl = slice("2023-06-01", "2023-06-30")
fig, ax = plt.subplots(2, 1, figsize=(11, 5))
sub[sl].plot(ax=ax[0], color="#1F3864", lw=.9)
ax[0].set_title("log-demand — level, June 2023 (mean drifts, variance drifts)")
ax[0].set_ylabel("log MW")
ds[sl].plot(ax=ax[1], color="#2E7D32", lw=.6)
ax[1].axhline(0, color="#C00000", lw=1)
ax[1].set_title(f"after D={diffs['D']}, d={diffs['d']} at m=24 — flat mean, stable spread")
ax[1].set_ylabel("seasonal difference")
plt.tight_layout()

# The over-differencing ladder: what each EXTRA regular difference would cost.
rows = []
for extra_d in (0, 1, 2):
    s = difference(sub, extra_d, 1, m=24)
    rows.append({"d": extra_d, "D": 1, "n": len(s),
                 "sd": round(float(s.std()), 5),
                 "variance": round(float(s.var()), 6),
                 "lag1_acf": round(float(acf(s, nlags=1, fft=True)[1]), 3)})
print(pd.DataFrame(rows).to_string(index=False))
print()
print("over-differencing signature: lag-1 ACF strongly negative (< -0.5) and variance rising")

**Expected**

```
 d  D    n      sd  variance  lag1_acf
 0  1 8736 0.05946  0.003535     0.898
 1  1 8735 0.02680  0.000718    -0.065
 2  1 8734 0.03910  0.001529    -0.508
```

Read the ladder honestly, because it does not say quite what the textbook slogan says.

- At `d=0` (our choice) the lag-1 ACF is **+0.898**. That is a lot of structure — and it is
  structure an AR term should *estimate*, not structure a difference should destroy. The
  tests already say stationary; we stop.
- At `d=1` the variance actually falls. This is the trap: the extra difference looks
  harmless, even flattering, so people take it. It is still a difference spent on nothing,
  and it converts an AR(1)-shaped series into one that needs an MA term to describe the
  same information.
- At `d=2` the damage is unmistakable: lag-1 ACF **−0.508** and the variance more than
  doubles, from 0.000718 to 0.001529. That is over-differencing in its textbook form —
  artificial negative autocorrelation injected at lag 1, and a noisier series than we
  started with. The `sim-overdiff` branch sits here, at roughly −0.6.

Every difference costs you an observation and adds an MA root you then have to model.
Difference as little as the tests allow.

### Task 4 — ACF and PACF: the model fingerprint

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from tayyar.analysis.correlograms import correlogram_table, suggest_orders, FINGERPRINT

fig, ax = plt.subplots(2, 2, figsize=(11, 6))
plot_acf(sub.dropna(), lags=72, ax=ax[0, 0])
ax[0, 0].set_title("ACF — level: slow, near-linear decay = non-stationary")
plot_pacf(sub.dropna(), lags=72, ax=ax[0, 1])
ax[0, 1].set_title("PACF — level: one huge spike at lag 1")
plot_acf(ds, lags=72, ax=ax[1, 0])
ax[1, 0].set_title(f"ACF — after D={diffs['D']}, d={diffs['d']}, m=24")
plot_pacf(ds, lags=72, ax=ax[1, 1])
ax[1, 1].set_title("PACF — after differencing")
for a in ax.ravel():
    a.axvline(24, color="#C00000", ls="--", lw=1)
    a.axvline(48, color="#C00000", ls=":", lw=.8)
    a.set_xlabel("lag (hours)")
ax[1, 0].annotate("m = 24", xy=(24, ax[1, 0].get_ylim()[1] * 0.7),
                  color="#C00000", fontsize=9)
plt.tight_layout()

tbl = correlogram_table(ds, nlags=72)
show = tbl[tbl.lag.isin([1, 2, 3, 4, 5, 23, 24, 25, 48, 72])]
print(show.round(3).to_string(index=False))
print()
print("suggest_orders:", suggest_orders(tbl, m=24))
print()
print(FINGERPRINT.to_string(index=False))

**Expected**

```
 lag    acf   pacf  acf_significant  pacf_significant
   1  0.898  0.899             True              True
   2  0.810  0.015             True             False
   3  0.732  0.006             True             False
   4  0.661 -0.001             True             False
   5  0.602  0.029             True              True
  23 -0.146 -0.089             True              True
  24 -0.208 -0.099             True              True
  25 -0.179  0.402             True              True
  48 -0.264 -0.072             True              True
  72  0.002 -0.023            False              True

suggest_orders: {'p': 5, 'q': 5, 'P': 1, 'Q': 1, 'm': 24}
```

**Reading the regular part.** The ACF decays geometrically (0.898, 0.810, 0.732, 0.661, …)
while the PACF collapses from 0.899 to 0.015 at lag 2. ACF tails off, PACF cuts off after
lag 1: that is the AR fingerprint, `p = 1`, possibly `p = 2` if you want a little slack.

**Reading the seasonal part.** After a seasonal difference the ACF is negative around
lag 24 (−0.208) and at 48 (−0.264), with a sharp PACF spike at lag 25 (0.402). A negative
ACF at the seasonal lag after seasonal differencing is the classic seasonal MA signature:
`Q = 1`, `P = 0` or 1.

**Why `suggest_orders` said `p=5, q=5`.** It returns the largest *significant* lag inside
1–5, and with 8,736 observations the significance band is ±0.021 — a PACF value of 0.029 at
lag 5 clears it while being of no practical interest. This is exactly why the function's
docstring calls it "a first candidate, never a verdict". Automation proposes; the analyst
disposes, and then confirms with residual diagnostics.

**Candidate orders**, to be settled by AIC/BIC and Ljung-Box in Lab 3:

| candidate | why |
|---|---|
| `SARIMA(1,0,1)(0,1,1)[24]` | AR(1) from the PACF cut-off, seasonal MA(1) for the ACF trough at 24 — the default first fit |
| `SARIMA(2,0,0)(1,1,0)[24]` | the pure-AR reading, if you take the marginal PACF spike at lag 5 seriously |
| `SARIMA(1,0,0)(1,1,1)[24]` | the parsimonious one; if it is within ~2 AIC of the others, ship it |

All three keep `d=0` and `D=1`. Never compare their AIC against a model with a different
`d` — the likelihoods are computed on different data.

### Mini-quiz

1. **ADF p = 0.01 — stationary or not?** Evidence of *stationary*: reject the unit-root null.
2. **KPSS p = 0.01 — stationary or not?** Evidence of *non-stationary*: reject the stationarity null.
3. **Seasonal or regular differencing first?** Seasonal, then re-test — it often removes the
   trend as well, as it did here.
4. **ACF tails off, PACF cuts off after lag 2?** AR(2).
5. **A slowly, linearly decaying ACF?** A trend — the series is non-stationary. Difference it
   before reading the correlogram any further; that shape is not a high AR order.

### Task 5 — `STATIONARITY.md`

```markdown
# Stationarity report — Tayyar log-demand, 2023

## 1. Verdict on the level series
ADF  stat = -3.103, p = 0.0263  -> reject unit root, evidence of stationarity
KPSS stat =  5.258, p = 0.01    -> reject stationarity, evidence of non-stationarity
VERDICT: difference-stationary. The level moves across the year even though the daily
cycle keeps pulling the series back to a local mean. Remedy is a difference, not a detrend.

## 2. Chosen differencing
d = 0, D = 1, m = 24.
Trace: (d=0, D=0) ADF 0.0263 / KPSS 0.01 -> difference-stationary
       (d=0, D=1) ADF 0.0000 / KPSS 0.10 -> stationary  <- stop here
One seasonal difference removed both the daily cycle and the slow level movement.
No regular difference is required; the reflexive d=1 would be a difference spent on nothing.

## 3. Evidence against over-differencing
d=0: sd 0.05946, var 0.003535, lag-1 ACF +0.898
d=1: sd 0.02680, var 0.000718, lag-1 ACF -0.065
d=2: sd 0.03910, var 0.001529, lag-1 ACF -0.508   <- variance rising, ACF turning negative
The +0.898 at d=0 is AR structure for the model to estimate, not a reason to difference again.

## 4. Candidate SARIMA orders
SARIMA(1,0,1)(0,1,1)[24] - ACF tails off geometrically, PACF cuts off after lag 1 (AR(1));
                           ACF negative at lags 24 and 48 after seasonal differencing (seasonal MA(1)).
SARIMA(2,0,0)(1,1,0)[24] - pure-AR reading, taking the marginal PACF spike at lag 5.
SARIMA(1,0,0)(1,1,1)[24] - parsimonious; prefer it if within ~2 AIC of the others.
All three share d=0, D=1 so their AIC values are comparable with each other and with nothing else.
Confirm the winner with Ljung-Box on the residuals, not with AIC alone.
```

```
git add -A && git commit -m "feat(analysis): stationarity report + candidate SARIMA orders"
```

### Instructor notes

- The `(d=0, D=1)` result is the lesson of this lab. Ask for the prediction in writing before
  Task 2 runs; most of the room writes `d=1, D=1` and the gap between the prediction and the
  trace is what makes the "difference as little as possible" rule stick.
- Branch `sim-overdiff` differences twice where once suffices. Do not announce it — give the
  pair the series and let them diagnose it from the lag-1 ACF near −0.6 and the inflated
  variance. The fix is to drop back one `d`, not to add an MA term to compensate.
- The statsmodels KPSS warning "p-value is greater than the indicated p-value" is **expected**
  and means p > 0.1. It is not an error, and the `0.01` at the other end means p < 0.01.
  Every cohort loses a few minutes to this warning unless you call it out first.
- `lags=20` is the default in every plotting API in the stack and **never reaches lag 24**.
  A pair that reports "no seasonal structure" has almost certainly left the default in place.
- Run the ACF/PACF speed round after Task 4: six anonymised correlogram pairs (AR, MA, ARMA,
  seasonal, white noise, non-stationary), 30 seconds each, called out loud. It is the fastest
  way to turn the fingerprint table from something read into something recognised.
- Fast finishers: run `choose_differencing` on the raw (un-logged) series and on 2021 alone,
  and explain why the verdict can change with the window. Regime shifts are why this report is
  re-run on a schedule in production, not written once.